# Final Proof of Concept
This notebook is the proof of concept of should work the ai agent pipeline with the Retrieval Augmented Generation System with ChromaDB and also with the Message Templates it would have.

### Import Librarys

In [1]:
from pathlib import Path
from ollama import chat
from langchain_ollama import ChatOllama
from langchain_ollama import OllamaEmbeddings
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

### Database setup

In [ ]:
project_path = next(
    folder for folder in (Path.cwd(), *Path.cwd().parents)
    if (folder / "Documents").is_dir() and (folder / "Notebooks").is_dir()
)

#---------------------------------------------------------

embeddings = OllamaEmbeddings(model="embeddinggemma")

vector_db = Chroma(
    collection_name="BMW_Models",
    embedding_function=embeddings,
    persist_directory=str(project_path / "chroma_db")
)

path_md_models = project_path / "Documents"

#---------------------------------------------------------

documents = []

for markdown in path_md_models.glob("*.md"):
    text = markdown.read_text(encoding="utf-8")

    document = Document(
        page_content=text,
        metadata={
            "source": str(markdown),
            "model": markdown.stem
        }
    )

    documents.append(document)

if not documents:
    raise ValueError("No documents found in the specified directory.")

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)

chunks = splitter.split_documents(documents)

#---------------------------------------------------------

# Clean old data.
existing_ids = vector_db.get(include=[])["ids"]

if existing_ids:
    vector_db.delete(ids=existing_ids)

# Load new fragments.
ids = vector_db.add_documents(chunks)

print("Total in Chroma:", len(vector_db.get(include=[])["ids"]))

print("Documents added to the vector database:", len(ids))


Total en Chroma: 381
Documents added to the vector database: 381


### Policy Settings

In [3]:
llm = ChatOllama(model='gemma4:e2b-it-q4_K_M',
                 temperature=0.5,
                 k_top_p=0.7,
                 num_ctx=8192,
                 num_predict=4096
                 )

#---------------------------------------------------------

